# FPL API Ingestion
Fetches bootstrap-static, element-summary, and fixtures from the FPL API
and writes JSON snapshots to `/Volumes/fpl/bronze/landing/`.

In [ ]:
%pip install aiohttp requests --quiet

In [ ]:
import asyncio, json, os
from datetime import datetime, timezone
from pathlib import Path
import aiohttp, requests

BASE_URL  = os.environ.get('FPL_API_BASE_URL', 'https://fantasy.premierleague.com/api/')
LANDING   = Path('/Volumes/fpl/bronze/landing')
TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H%M%SZ')

def _write(sub_path, data):
    dest = LANDING / sub_path
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(data)
    print(f'Written: {dest}')

def fetch_bootstrap():
    r = requests.get(f'{BASE_URL}bootstrap-static/', timeout=30)
    r.raise_for_status()
    return r.json()

def fetch_fixtures():
    r = requests.get(f'{BASE_URL}fixtures/', timeout=30)
    r.raise_for_status()
    return r.json()

async def _fetch_element(session, eid):
    url = f'{BASE_URL}element-summary/{eid}/'
    try:
        async with session.get(url, timeout=aiohttp.ClientTimeout(total=20)) as r:
            r.raise_for_status()
            return eid, await r.json()
    except Exception as e:
        print(f'Warning: element {eid} failed: {e}')
        return eid, None

async def fetch_all_elements(ids):
    async with aiohttp.ClientSession() as session:
        return await asyncio.gather(*[_fetch_element(session, i) for i in ids])

In [ ]:
print('Fetching bootstrap-static...')
bootstrap = fetch_bootstrap()
_write(f'bootstrap-static/bootstrap-static_{TIMESTAMP}.json', json.dumps(bootstrap).encode())

print('Fetching fixtures...')
_write(f'fixtures/fixtures_{TIMESTAMP}.json', json.dumps(fetch_fixtures()).encode())

ids = [e['id'] for e in bootstrap['elements']]
print(f'Fetching {len(ids)} element summaries...')
results = asyncio.run(fetch_all_elements(ids))
payload = [{'element_id': eid, **data} for eid, data in results if data]
_write(f'element-data/element_data_{TIMESTAMP}.json', json.dumps(payload).encode())
print(f'Done — {len(payload)} element summaries written.')